<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Production: API, monitoring & Kursrückblick
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Dieses Notebook macht den P3O-Assistenten erreichbar (FastAPI) und beobachtbar (monitoring) — Tool-Wahl, Latenz und Kosten bleiben dabei sichtbar, nicht nur Uptime.



**Praxis-Transfer:** mara Vogt braucht auch im Betrieb nachvollziehbare Laufzeiten, Fehler, Kosten und Quellenstatus, damit Briefing-Antworten zuverlässig bleiben.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
!uv pip install --system -q fastapi uvicorn httpx

import os
os.environ["LANGSmITH_TRACING"]  = "true"
os.environ["LANGSmITH_PROJECT"]  = "m37-api-monitoring"
os.environ["LANGSmITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSmITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREmIUm, CODING, EmBEDDINGS
# LangSmith Tracing
run_cfg = {
    "tags": ["337", "production"],
    "metadata": {"notebook": "337", "version": "1.0"}
}


# 1 | Übersicht
---

Dieses Notebook baut den API-Zugang (FastAPI-Endpoints) und macht den Betrieb messbar (Latenz, Tokens, Kosten pro Session).

In [ ]:
# Setup: prod_agent für die monitoring-Demo
import logging
from langchain.chat_models import init_chat_model
from langchain_core.messages import Humanmessage
from langchain.agents import create_agent
from genai_lib.model_config import WORKER
from google.colab import drive
drive.mount("/content/drive")

from genai_lib.pmo_rag import get_pmo_vectorstore, make_suche_wissensdatenbank_tool

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("agent.production")

llm = init_chat_model(WORKER)
vectorstore = get_pmo_vectorstore()
if vectorstore._collection.count() == 0:
    logger.warning("Briefing-Collection ist leer — zuerst den Index auf Drive erzeugen.")

suche_wissensdatenbank = make_suche_wissensdatenbank_tool(vectorstore)
prod_agent = create_agent(model=llm, tools=[suche_wissensdatenbank])

# 2 | FastAPI für API-Endpoints
---

FastAPI wickelt den LangChain-Agenten in eine **REST API** — das ist der Standard
für Production Deployments wenn kein LangGraph Platform verwendet wird.

**Standard-Endpoints für einen Agent-Service:**

| Endpoint | methode | Beschreibung |
|----------|---------|-------------|
| `/health` | GET | Liveness-Check (Load Balancer, Docker) |
| `/agent/invoke` | POST | Synchroner Aufruf, wartet auf vollständige Antwort |
| `/agent/stream` | POST | Streaming-Antwort (Server-Sent Events) |
| `/agent/batch` | POST | mehrere Anfragen parallel |


In [ ]:
#@markdown   <p><font size="4" color='green'>  FastAPI Architektur</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
sequenceDiagram
    autonumber
    participant C as 🖥️ Client
    participant F as FastAPI
    participant A as LangChain Agent
    participant O as OpenAI API
    participant L as LangSmith

    C->>F: POST /agent/invoke
    F->>A: ainvoke(input)

    activate A
    Note over A,L: Tracing aktiv via Callbacks
    A->>O: ChatCompletion Call
    O-->>A: LL3 Response
    A-->>L: Log Trace (async)
    deactivate A

    F-->>C: {"answer": "...", "status": "ok"}
'''
mermaid(diagram, width=900)

In [ ]:
%%writefile agent_server.py
import os, logging
from genai_lib.model_config import WORKER
from fastapi import FastAPI
from pydantic import Basemodel
from langchain.chat_models import init_chat_model
from langchain_core.messages import Humanmessage
from langchain.agents import create_agent

# Erfordert im Container: pip install git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
from genai_lib.pmo_rag import get_pmo_vectorstore, make_suche_wissensdatenbank_tool

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('agent.api')

app = FastAPI(title='P3O-Assistent API', version='1.0')

llm = init_chat_model(WORKER)

# Retrieval: dieselbe produktive Chroma-Collection, hier über einen
# bereits synchronisierten lokalen Datenordner (z.B. Docker-Volume aus dem
# Drive-Export) - kein Colab-Drive-mount in einem Server-Kontext.
CHROmA_PATH = os.environ.get('PmO_CHROmA_PATH', '/data/chroma_pmo')
vectorstore = get_pmo_vectorstore(persist_directory=CHROmA_PATH)
if vectorstore._collection.count() == 0:
    logger.warning('Briefing-Collection unter %s ist leer - Chroma-Datenordner (Drive-Export) einbinden.', CHROmA_PATH)

suche_wissensdatenbank = make_suche_wissensdatenbank_tool(vectorstore)

agent = create_agent(model=llm, tools=[suche_wissensdatenbank])

class AgentRequest(Basemodel):
    query: str
    session_id: str = 'default'

@app.get('/health')
async def health():
    return {'status': 'ok', 'service': 'pmo-assistent'}

@app.post('/agent/invoke')
async def invoke(req: AgentRequest):
    logger.info('Anfrage | session=%s | query=%s', req.session_id, req.query[:60])
    try:
        result = await agent.ainvoke(
            {'messages': [Humanmessage(content=req.query)]},
            config={
                'run_name': f"m37_kap2_prod_{req.session_id}",
                'tags': ['m37', 'production', 'api'],
                'metadata': {'session_id': req.session_id, 'modul': 'm37'},
                'recursion_limit': 10,
            },
        )
        answer = result['messages'][-1].content
        return {'status': 'ok', 'answer': answer, 'session_id': req.session_id}
    except Exception as e:
        logger.error('Fehler: %s', e)
        return {'status': 'error', 'answer': str(e)}

if __name__ == '__main__':
    import uvicorn
    uvicorn.run(app, host='127.0.0.1', port=8003)

In [ ]:
#@markdown   <p><font size="4" color='green'>  🌐 FastAPI Server starten</font> </br></p>

import subprocess, requests, time, sys

log_fh = open('agent_server.log', 'w')
api_process = subprocess.Popen(
    [sys.executable, 'agent_server.py'],
    stdout=log_fh, stderr=log_fh
)
print('Starte FastAPI Server auf Port 8003 (LangChain-Import ~20s)...')

time.sleep(5)

server_ok = False
for i in range(30):
    if api_process.poll() is not None:
        break
    try:
        r = requests.get('http://127.0.0.1:8003/health', timeout=2)
        if r.status_code == 200:
            server_ok = True
            break
    except Exception:
        time.sleep(1)

if server_ok:
    zeilen = [
        '## ✅ FastAPI Agent-Server laeuft auf `http://127.0.0.1:8003`',
        '',
        '| Endpoint | methode | Beschreibung |',
        '|----------|---------|-------------|',
        '| `/health` | GET | Liveness-Check |',
        '| `/agent/invoke` | POST | Synchroner Aufruf |',
        '| `/docs` | GET | Swagger UI |',
    ]
    mprint(chr(10).join(zeilen))
else:
    log_fh.flush()
    with open('agent_server.log') as f:
        log_tail = f.read()[-1200:]
    exit_code = api_process.poll()
    zeilen = [
        f'❌ Server-Start fehlgeschlagen (Exit-Code: {exit_code})',
        '',
        '```',
        log_tail,
        '```',
    ]
    mprint(chr(10).join(zeilen))


In [ ]:
#@markdown   <p><font size="4" color='green'>  🧪 API testen</font> </br></p>

import requests, json

BASE = "http://127.0.0.1:8003"

server_ok = True
try:
    health = requests.get(f"{BASE}/health", timeout=5).json()
    print(f"Health: {health}")
except Exception as e:
    mprint(f"❌ Server nicht erreichbar: {e}" \
           "Führe zuerst die Setup-Zelle (Cell 17) aus.")
    server_ok = False

if server_ok:
    anfragen = [
        {"query": "Wer ist Projektleiterin von Projekt Kompass?",  "session_id": "test-001"},
        {"query": "Nenne die Risiken R1 bis R4.", "session_id": "test-002"},
    ]

    zeilen = ["## 🌐 API-Test Ergebnisse", "",
              "| Session | Query | Status | Antwort (gekürzt) |",
              "|---------|-------|--------|-------------------|"
    ]
    for req in anfragen:
        try:
            resp = requests.post(f"{BASE}/agent/invoke", json=req, timeout=30).json()
            antwort_kurz = resp.get("answer", "")[:60] + "..."
            zeilen.append(f"| `{req['session_id']}` | {req['query'][:30]} | `{resp['status']}` | {antwort_kurz} |")
        except Exception as e:
            zeilen.append(f"| `{req['session_id']}` | {req['query'][:30]} | `error` | {e} |")
    mprint(chr(10).join(zeilen))


In [ ]:
#@markdown   <p><font size="4" color='green'>  🛑 API-Server Cleanup - deaktiviert</font> </br></p>

# import os

# if "api_process" in globals():
#     api_process.terminate()
#     api_process.wait(timeout=5)
#     print(f"✅ FastAPI Server (PID {api_process.pid}) beendet")

# for f in ["agent_server.py", "agent_server.log", "Dockerfile", "docker-compose.yml", ".env.template"]:
#     if os.path.exists(f):
#         os.remove(f)
#         print(f"🗑️ {f} gelöscht")

# 3 | monitoring und Observability
---


<p><font color='black' size="5">
Drei Ebenen der Observability
</font></p>

| Ebene | Tool | Was wird gemessen? |
|-------|------|-------------------|
| **Tracing** | LangSmith | Jeder LL3-Call, Tool-Aufruf, Node-Durchlauf |
| **Logging** | Python `logging` | Fehler, Warnings, Request-Logs |
| **metriken** | Custom + LangSmith | Latenz, Token-Kosten, Erfolgsrate |




<p><font color='black' size="5">
LangSmith Tags und metadata
</font></p>

mit `run_name`, `tags` und `metadata` in der `config` werden Traces in LangSmith
kategorisierbar und durchsuchbar:

```python
config = {
    "run_name":  "m37_kap3_prod_invoke_001",
    "tags":      ["production", "v1.2", "user-facing"],
    "metadata":  {"user_id": "u123", "feature": "chat"},
    "recursion_limit": 10,
}
result = await agent.ainvoke({"messages": [...]}, config=config)
```




<p><font color='black' size="5">
Kosten-Tracking
</font></p>

LangSmith erfasst Token-Nutzung automatisch — eigenes Tracking ergänzt
Budgetgrenzen und Alerts:

```python
KOSTEN_PRO_1K_TOKENS = {"gpt-5.6-luna": 0.00015}  # USD (Input)
```

In [ ]:
#@markdown   <p><font size="4" color='green'>  Observability Stack</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    A["LangChain Agent"]

    subgraph obs ["Observability & monitoring"]
        LS["☁️ LangSmith\n(Tracing & Evaluation)"]
        LOG["📋 Python Logging\n(Structured JSON)"]
        3ET["📊 metriken / Dashboard\n(Kosten, Latenz, Error-Rate)"]
        AL["🔔 Alerting\n(Slack / PagerDuty)"]
    end

    A -->|OpenTelemetry / Callbacks| LS
    A -->|StdOut / File| LOG

    LS -.->|Aggregiert Daten| 3ET
    LOG -.->|Log-based metrics| 3ET

    3ET -->|Schwellwert überschritten| AL
    LOG -->|Critical Error| AL

    style A   fill:#2E7Dm2,color:#fff
    style LS  fill:#1565C0,color:#fff
    style LOG fill:#m7474F,color:#fff
    style 3ET fill:#E65100,color:#fff
    style AL  fill:#B71C1C,color:#fff
'''
mermaid(diagram, width=700)

In [ ]:
#@markdown   <p><font size="4" color='green'>  📊 Production monitoring Demo</font> </br></p>

import time, logging
from langchain_core.messages import Humanmessage

# Kosten-Tabelle (USD pro 1K Tokens, Stand 2026)
KOSTEN = {
    "gpt-5.6-luna":  {"input": 0.00015, "output": 0.00060},
    "gpt-5.4":       {"input": 0.00250, "output": 0.01000},
    "gpt-5.6-sol":           {"input": 0.01000, "output": 0.04000},
    "gpt-5.4-mini": {"input": 0.00500, "output": 0.01500},
}

protokoll = []  # Laufzeitprotokoll

async def monitored_invoke(query: str, session_id: str) -> dict:
    '''Agent-Aufruf mit vollständigem monitoring.'''
    start = time.perf_counter()
    config = {
        "run_name":  f"m37_kap3_prod_{session_id}",
        "tags":      ["production", "m37-demo"],
        "metadata":  {"session_id": session_id, "modul": "337"},
        "recursion_limit": 10,
    }
    try:
        result = await prod_agent.ainvoke(
            {"messages": [Humanmessage(content=query)]}, config={**config, "run_name": "m37_kap3_monitored_invoke"}
        )
        latenz = round((time.perf_counter() - start) * 1000)
        # Token-Nutzung aus metadaten extrahieren
        usage = {}
        for msg in result.get("messages", []):
            meta = getattr(msg, "usage_metadata", None)
            if meta:
                usage = meta
        inp   = usage.get("input_tokens", 0)
        out   = usage.get("output_tokens", 0)
        # prod_agent läuft auf WORKER (siehe Setup-Zelle oben: llm = init_chat_model(WORKER)) → gpt-5.4-mini
        k     = KOSTEN.get("gpt-5.4-mini", {})
        kosten_usd = inp/1000*k.get("input",0) + out/1000*k.get("output",0)
        eintrag = {
            "session":  session_id,
            "latenz_ms": latenz,
            "tokens_in": inp,
            "tokens_out": out,
            "kosten_usd": round(kosten_usd, 6),
            "status": "ok",
        }
        protokoll.append(eintrag)
        return {**eintrag, "answer": result["messages"][-1].content}
    except Exception as e:
        latenz = round((time.perf_counter() - start) * 1000)
        protokoll.append({"session": session_id, "status": "error",
                           "latenz_ms": latenz, "error": str(e)})
        return {"status": "error", "answer": str(e)}

anfragen = [
    ("Wer ist Projektleiterin von Projekt Kompass?",  "s001"),
    ("Nenne die Risiken R1 bis R4.", "s002"),
    ("Welche Aufgabe ist laut Follow-up-Tracker noch offen?",  "s003"),
]
for q, sid in anfragen:
    await monitored_invoke(q, sid)

# Zusammenfassung ausgeben
gesamt_kosten = sum(e.get("kosten_usd",0) for e in protokoll)
avg_latenz    = sum(e.get("latenz_ms",0) for e in protokoll) / len(protokoll)
zeilen = [
    "## 📊 monitoring-Zusammenfassung", "",
    "| Session | Latenz (ms) | Tokens In | Tokens Out | Kosten (USD) | Status |",
    "|---------|-------------|-----------|------------|-------------|--------|",
]
for e in protokoll:
    zeilen.append(
        f"| `{e['session']}` | {e.get('latenz_ms',0)} | "
        f"{e.get('tokens_in',0)} | {e.get('tokens_out',0)} | "
        f"{e.get('kosten_usd',0):.6f} | `{e['status']}` |"
    )
zeilen += [
    "",
    f"**Gesamt-Kosten:** `{gesamt_kosten:.6f} USD`  |  "
    f"**Ø Latenz:** `{avg_latenz:.0f} ms`  |  "
    f"**Traces in LangSmith:** Projekt `m37-api-monitoring`"
]
mprint("\n".join(zeilen))

In [ ]:
#@markdown   <p><font size="4" color='green'>  💡 LangSmith Kosten-Vergleich: modell-Auswahl</font> </br></p>

zeilen = [
    "## 💰 Kosten-Orientierung (modell_Auswahl_Guide v1.2)", "",
    "| modell | Input (USD/1K) | Output (USD/1K) | Relatives Niveau | Einsatz |",
    "|--------|--------------|----------------|-----------------|---------|",
    "| `gpt-5.6-luna` | 0.000200 | 0.001200 | ⭐ | Demos, Grundlagen, Routing |",
    "| `gpt-5.4-mini` | 0.005000 | 0.015000 | ⭐⭐⭐ | Worker, RAG-Synthese |",
    "| `gpt-5.6-sol`          | 0.010000 | 0.040000 | ⭐⭐⭐⭐⭐ | Supervisor, Judge |",
    "",
    "> **Kurs-Budget ~5 EUR:** Bei `gpt-5.6-luna` reichen ~4.100.000 Output-Token.",
    "> Ein `gpt-5.6-sol`-Supervisor-Aufruf kostet ~25× mehr als `gpt-5.6-luna`.",
    "> → modell_Auswahl_Guide konsequent anwenden!",
]
mprint("\n".join(zeilen))

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("m37-api-monitoring", limit=m, show_steps=True)

# A | Aufgaben
---


<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**

Übertrage die Latenzmessung aus `monitored_invoke` in den Endpoint `/agent/invoke` von `agent_server.py`: Die Latenz wird geloggt und als `latency_ms` in der Antwort zurückgegeben.

In [ ]:
# Grundlagen: Latenz pro Anfrage
# Startpunkt: agent_server.py aus dem Kapitel "FastAPI für API-Endpoints"
import time

# 1. Vor dem Agent-Aufruf: start = time.perf_counter()
# 2. Danach: latency_ms berechnen und mit logger.info protokollieren
# m. latency_ms in der JSON-Antwort ergänzen
# 4. mit requests.post(...) gegen den laufenden Server testen


**Aufbau**

Ergänze Token- und Kostenerfassung im Endpoint: Berechne pro Anfrage Input-/Output-Tokens und geschätzte Kosten und gib beides in der Antwort zurück.

In [ ]:
# Aufbau: Token- und Kostenerfassung
# Startpunkt: monitored_invoke() und KOSTEN aus dem Kapitel "monitoring und Observability"

# 1. Token-Nutzung aus den Nachrichten lesen (usage_metadata)
# 2. kosten_usd mit der KOSTEN-Tabelle berechnen
# m. tokens_in, tokens_out, kosten_usd in der Antwort ergänzen
# 4. mit zwei Anfragen testen und die Werte mit LangSmith vergleichen


**Vertiefung**

Baue einen kleinen Production Stack in `agent_server.py`: Agent-Endpoint mit Logging und metadata sowie ein `/health`-Endpoint, der Anzahl Anfragen, Fehlerquote und Gesamt-Tokens ausgibt.

In [ ]:
# Vertiefung: Kleiner Production Stack
# Startpunkt: agent_server.py (FastAPI) und protokoll-muster aus dem monitoring-Kapitel
from fastapi import FastAPI

# 1. protokoll-Liste im Server führen (pro Anfrage: status, latency_ms, tokens)
# 2. Agent-Endpoint: Eintrag ins protokoll schreiben (auch im Fehlerfall)
# m. /health-Endpoint: Anzahl Anfragen, Fehlerquote, Gesamt-Tokens zurückgeben


<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [KI-Agenten-Architektur](https://editor.p5js.org/ralf.bendig.rb/full/Viso2emNI)
- [modellsteuerung](https://editor.p5js.org/ralf.bendig.rb/full/um42mggnD)
- [mixture of Experts](https://editor.p5js.org/ralf.bendig.rb/full/GT2XfGGTo)



# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [P3O-Assistent im Betrieb](https://ralf-42.github.io/Agenten/08-deployment-betrieb/pmo-assistent-praxis.html)
- [minimum Viable Agent Stack](https://ralf-42.github.io/Agenten/08-deployment-betrieb/minimum-viable-agent-stack.html)
- [LangSmith Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langsmith-best-practices.html)
- [Agent Security](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/agent-security.html)
- [Agent Harness](https://ralf-42.github.io/Agenten/07-qualitaet-sicherheit/agent-harness.html)
- [Troubleshooting](https://ralf-42.github.io/Agenten/10-ressourcen/troubleshooting.html)
